# 12B Appendix: Reading the Singular Vectors


[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tools4ds/DS701-Materials-FA26/blob/main/jupyter_notebooks/12B-Appendix-Reading-Singular-Vectors.ipynb)

[Lecture 12](12-SVD-Low-Rank.qmd) gives two readings of a truncated SVD $A^{(k)} = U'\Sigma'(V')^T$: the columns of $U$ are __common patterns__, and the rows of $U'\Sigma'$ and columns of $(V')^T$ are __latent factors__. Its examples (a week of network traffic, the Netflix prize) are too large to read entry by entry. This appendix works through one small table with labeled rows and columns, and shows how to say in words what $\mathbf{u}_1, \mathbf{v}_1, \mathbf{u}_2, \mathbf{v}_2$ mean.

Four things to take away:

1. For a table of nonnegative numbers, $\mathbf{v}_1$ and $\mathbf{u}_1$ each have all entries of one sign.
   * $\mathbf{v}_1$ has one entry per column. It is a "typical row" of $A$: not any actual row, but the shape, scaled to length 1, that every row roughly follows.
   * $\mathbf{u}_1$ has one entry per row. It scores each row by *how much* of the typical row it has. Read down the columns instead, and $\mathbf{u}_1$ is also the "typical column" of $A$: not any actual column, but the shape, scaled to length 1, that every column roughly follows.
2. Scaled by $\sigma_1$, row $i$'s score is its first coordinate, $\sigma_1 u_{i1}$. It is the dot product $\mathbf{a}_i \cdot \mathbf{v}_1$, where $\mathbf{a}_i$ is row $i$ of $A$: a *weighted* total of the row.
3. $\mathbf{v}_2$ must be perpendicular to $\mathbf{v}_1$, so it *must* have mixed signs. It is a contrast between two groups of columns, and $\mathbf{u}_2$ says which side each row is on.
4. Every $\mathbf{v}_k$ is labeled by the columns of $A$, and every $\mathbf{u}_k$ by the rows.

The derivations of the SVD itself are in [Appendix 12A](12A-Appendix-SVD-Derivations.qmd).

## The Example: Hours of Streaming by Genre

A streaming service records how many hours six subscribers spent on four genres last month:

| | action | sci-fi | romance | drama | total |
|:--|:--:|:--:|:--:|:--:|:--:|
| Ana | 9 | 6 | 1 | 0 | 16 |
| Ben | 7 | 8 | 0 | 1 | 16 |
| Cleo | 1 | 0 | 8 | 8 | 17 |
| Dev | 0 | 2 | 7 | 6 | 15 |
| Eli | 2 | 2 | 2 | 2 | 8 |
| Fay | 5 | 4 | 5 | 4 | 18 |

The table was made to have a structure you can see by eye: Ana and Ben watch action and sci-fi, Cleo and Dev watch romance and drama, and Eli and Fay watch everything evenly, Eli lightly and Fay heavily. The question is whether the SVD finds the same structure, and where in $U$, $\Sigma$, $V$ it puts it.

In [ ]:
#| code-fold: false
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
np.set_printoptions(precision=2, suppress=True)

viewers = ['Ana', 'Ben', 'Cleo', 'Dev', 'Eli', 'Fay']
genres = ['action', 'sci-fi', 'romance', 'drama']
A = np.array([[9, 6, 1, 0],
              [7, 8, 0, 1],
              [1, 0, 8, 8],
              [0, 2, 7, 6],
              [2, 2, 2, 2],
              [5, 4, 5, 4]], dtype=float)

u, s, vt = np.linalg.svd(A, full_matrices=False)

# Singular vectors are only determined up to sign. Flip each pair (u_i, v_i)
# together so that the first entry of v_i is positive; sigma_i u_i v_i^T is unchanged.
for i in range(len(s)):
    if vt[i, 0] < 0:
        vt[i] *= -1
        u[:, i] *= -1

print("singular values:          ", s)
print("fraction of ||A||_F^2, cum:", np.cumsum(s**2) / np.sum(s**2))

$\Vert A\Vert_F^2 = 548$ and $\sigma_1^2 + \sigma_2^2 = 354.7 + 185.1 = 539.8$, so the first two terms carry $98.5\%$ of the table. $\sigma_3$ and $\sigma_4$ are small: they hold the noise of individual viewing habits (Ana's 9 versus 6 instead of 7.5 and 7.5). The table is effectively __rank 2__, and the whole story is in the first two singular-vector pairs.

## Which Labels Go with Which Vectors

The shapes tell you. $A$ is $6\times 4$: rows are viewers, columns are genres.

* Each $\mathbf{v}_k$ is in $\mathbb{R}^4$: __one entry per column__. Every right singular vector, not just $\mathbf{v}_1$, is indexed by the genres.
* Each $\mathbf{u}_k$ is in $\mathbb{R}^6$: __one entry per row__. Every left singular vector is indexed by the viewers.

**Notation: rows and columns of $A$**

In this appendix $\mathbf{a}_i$ is __row__ $i$ of $A$, one viewer's hours as a vector in $\mathbb{R}^4$; for example $\mathbf{a}_{\text{Ana}} = (9, 6, 1, 0)$. Lecture 12's "common patterns" reading used $\mathbf{a}_j$ for __column__ $j$. The subscript follows the usual convention: $i$ indexes rows, $j$ indexes columns.


To see why, read $A\mathbf{v}_k = \sigma_k\mathbf{u}_k$ one row at a time. Entry $i$ of $A\mathbf{v}_k$ is the inner product of row $i$ of $A$ with $\mathbf{v}_k$:

$$
\sigma_k u_{ik} = \mathbf{a}_i\cdot\mathbf{v}_k = \sum_{j=1}^{4} a_{ij}\,v_{kj} .
$$

* The inner product pairs entry $j$ of $\mathbf{v}_k$ with $a_{ij}$, viewer $i$'s hours in genre $j$. So $\mathbf{v}_k$ has exactly as many entries as $A$ has columns (here 4), and its $j$-th entry goes with column $j$, genre $j$.
* There is one inner product for each row of $A$. So $\sigma_k\mathbf{u}_k$ has exactly as many entries as $A$ has rows (here 6), and its $i$-th entry goes with row $i$, viewer $i$.

Put the labels on explicitly:

In [ ]:
#| code-fold: false
k = 2
V_k = pd.DataFrame(vt[:k].T, index=genres, columns=['v1', 'v2'])
US_k = pd.DataFrame(u[:, :k] * s[:k], index=viewers, columns=['sigma1 u1', 'sigma2 u2'])
V_k

In [ ]:
#| code-fold: false
US_k.assign(total_hours=A.sum(axis=1))

The first table is $V'$ (the first two columns of $V$). The second is $U'\Sigma'$: row $i$ is viewer $i$'s pair of coordinates $(\sigma_1 u_{i1},\ \sigma_2 u_{i2})$. Rounded:

$$
\mathbf v_1 = \begin{bmatrix}0.55\\0.49\\0.50\\0.45\end{bmatrix}
\begin{matrix}\text{action}\\\text{sci-fi}\\\text{romance}\\\text{drama}\end{matrix},
\qquad
\mathbf v_2 = \begin{bmatrix}0.52\\0.43\\-0.54\\-0.50\end{bmatrix},
\qquad
\sigma_1\mathbf u_1 = \begin{bmatrix}8.39\\8.23\\8.19\\7.21\\3.99\\9.03\end{bmatrix}
\begin{matrix}\text{Ana}\\\text{Ben}\\\text{Cleo}\\\text{Dev}\\\text{Eli}\\\text{Fay}\end{matrix},
\qquad
\sigma_2\mathbf u_2 = \begin{bmatrix}6.74\\6.59\\-7.81\\-5.93\\-0.18\\-0.37\end{bmatrix}.
$$

Reading $\sigma_k\mathbf{u}_k$ rather than $\mathbf{u}_k$ changes nothing about the *ordering* or *signs* within one column, since $\sigma_k > 0$ multiplies every entry. It only puts the numbers in the units of the data (hours), which makes them easier to interpret.

**Signs are arbitrary; read them as a pair**

Software may return $\mathbf{u}_1$ and $\mathbf{v}_1$ with every entry negative; the traffic example in Lecture 12 had exactly this. Flipping $\mathbf{u}_k$ and $\mathbf{v}_k$ *together* leaves $\sigma_k\mathbf{u}_k\mathbf{v}_k^T$, and therefore $A$, unchanged. So the absolute sign of $\mathbf{v}_2$ means nothing; what means something is which entries share a sign, and the product $u_{ik}v_{kj}$. That is why we fix a convention (first entry of each $\mathbf{v}_k$ positive) before reading anything.


# The First Pair: Overall Volume

## Why $\mathbf{v}_1$ has no negative entries

$\mathbf{v}_1$ is the unit vector that makes $\Vert A\mathbf{v}\Vert$ as large as possible (it is the top eigenvector of $A^TA$, [Appendix 12A](12A-Appendix-SVD-Derivations.qmd#where-do-u-sigma-v-come-from)). Written row by row,

$$
\Vert A\mathbf{v}\Vert^2 = \sum_{i=1}^m (\mathbf{a}_i\cdot\mathbf{v})^2,
$$

where $\mathbf{a}_i$ is row $i$ of $A$. So $\mathbf{v}_1$ is the single direction that is __best aligned with all the rows at once__.

Now suppose every entry of $A$ is $\geq 0$, as with hours, counts, purchases or pixel intensities. Take any unit vector $\mathbf{v}$ and replace each entry by its absolute value, giving $|\mathbf{v}|$. Because $\mathbf{a}_i$ has no negative entries,

$$
|\mathbf{a}_i\cdot\mathbf{v}| \;\leq\; \mathbf{a}_i\cdot|\mathbf{v}| \quad\text{for every row } i,
$$

so $|\mathbf{v}|$ does at least as well as $\mathbf{v}$. A maximizer can always be taken with no negative entries. (This is a special case of the Perron–Frobenius theorem.) Then $\mathbf{u}_1 = A\mathbf{v}_1/\sigma_1$ is a nonnegative combination of nonnegative columns, so it has no negative entries either.

Geometrically: every row lives in the positive corner of $\mathbb{R}^4$, and the direction that points at all of them at once points into that corner too. It is not a contrast; it is a __typical row__. Here $\mathbf{v}_1 = (0.55, 0.49, 0.50, 0.45)$ has roughly equal weight on all four genres: the typical subscriber's month is a bit of everything.

## The rank-1 reading: shared shape, individual amount

Keep only the first term, $A^{(1)} = \sigma_1\mathbf{u}_1\mathbf{v}_1^T$. Row $i$ of it is

$$
\text{row } i \text{ of } A^{(1)} = (\sigma_1 u_{i1})\,\mathbf{v}_1^T .
$$

Every row is the __same profile__ $\mathbf{v}_1^T$, scaled by __one number per row__. The first term can make one viewer watch more than another, but it cannot make one viewer prefer action and another romance. The only thing it can vary between rows is *how much*.

In [ ]:
#| code-fold: false
A1 = s[0] * np.outer(u[:, 0], vt[0])
pd.DataFrame(A1, index=viewers, columns=genres).round(1)

Every row has the same shape (action ≳ romance ≈ sci-fi ≳ drama), and the rows differ only in scale: Eli's row is about half of everyone else's. Ana and Cleo, who have opposite tastes, get almost the same row. The rank-1 approximation sees their volume and misses their preference entirely; that is the job of the second term. This is also why $\sigma_1^2$ is only $65\%$ of $\Vert A\Vert_F^2$ here: volume explains a lot, but not everything.

Read the same table down a column instead. Column $j$ of $A^{(1)}$ is $(\sigma_1 v_{1j})\,\mathbf{u}_1$, so every genre's column is a multiple of $\mathbf{u}_1$: the same pattern across viewers (Eli lowest, Fay highest), scaled by the weight $\mathbf{v}_1$ gives that genre. So $\mathbf{u}_1$ is the "typical column" just as $\mathbf{v}_1$ is the typical row. This is Lecture 12's "common patterns" reading: $\mathbf{u}_1$ is the strongest pattern that all the columns share.

## The first coordinate is a projection

The first column of $AV = U\Sigma$ reads $A\mathbf{v}_1 = \sigma_1\mathbf{u}_1$. Entry $i$ of that is

$$
\sigma_1 u_{i1} = \mathbf{a}_i\cdot\mathbf{v}_1 ,
$$

the dot product of row $i$ with $\mathbf{v}_1$. You can check it by hand. For Ana, $\mathbf{a}_{\text{Ana}} = (9, 6, 1, 0)$:

$$
\sigma_1 u_{\text{Ana},1} = 9(0.55) + 6(0.49) + 1(0.50) + 0(0.45) = 4.95 + 2.94 + 0.50 = 8.39 .
$$

In [ ]:
#| code-fold: false
print("A @ v1      :", A @ vt[0])
print("sigma1 * u1 :", s[0] * u[:, 0])

So a viewer's first coordinate is their __total viewing, weighted by $\mathbf{v}_1$__. Since $\mathbf{v}_1$ is close to uniform here, it is close to $0.5\times$ their total hours: Eli watches 8 hours and scores 3.99; Fay watches 18 and scores 9.03, the largest.

## A weighted total, not a raw total

Close to uniform is not uniform. $\mathbf{v}_1$ gives action ($0.55$) more weight than romance ($0.50$) or drama ($0.45$), because action is the genre most aligned with the rows overall. That is enough to reorder two viewers:

| | total hours | $\sigma_1 u_{i1}$ |
|:--|:--:|:--:|
| Ana | 16 | 8.39 |
| Cleo | 17 | 8.19 |

Cleo watches more, but Ana has the larger first coordinate, because Ana's hours are in the genres $\mathbf{v}_1$ weights more heavily. When you are asked who has the largest first coordinate, read it off $\mathbf{u}_1$ (or compute $\mathbf{a}_i\cdot\mathbf{v}_1$), not off the row totals. The [unequal-groups example](#when-the-groups-are-unequal) below makes this gap much larger.

# The Second Pair: A Contrast

## Why $\mathbf{v}_2$ must have mixed signs

The right singular vectors are orthonormal, so $\mathbf{v}_1\cdot\mathbf{v}_2 = 0$. Every entry of $\mathbf{v}_1$ is positive. If every entry of $\mathbf{v}_2$ had the same sign, $\mathbf{v}_1\cdot\mathbf{v}_2$ would be a sum of terms with the same sign, which cannot be zero. So for nonnegative data, __$\mathbf{v}_2$ is always a contrast__: some columns positive, some negative.

Read it by grouping columns by sign:

$$
\mathbf v_2 = \begin{bmatrix}+0.52\\+0.43\\-0.54\\-0.50\end{bmatrix}
\begin{matrix}\text{action}\\\text{sci-fi}\\\text{romance}\\\text{drama}\end{matrix}
$$

Action and sci-fi on one side, romance and drama on the other. $\mathbf{v}_2$ is the __action/sci-fi versus romance/drama__ preference. The name is yours to choose; the content is in the signs and the sizes. A column whose entry in $\mathbf{v}_2$ was near zero would not take part in the contrast.

## Reading $\mathbf{u}_2$: which side each row is on

By the same argument as before, $\sigma_2 u_{i2} = \mathbf{a}_i\cdot\mathbf{v}_2$: action and sci-fi hours count positive, romance and drama hours count negative. For Ana, $9(0.52) + 6(0.43) - 1(0.54) - 0(0.50) = 6.72 \approx 6.74$ (the difference is rounding in $\mathbf{v}_2$).

| | $\sigma_2 u_{i2}$ | reading |
|:--|:--:|:--|
| Ana | $+6.74$ | action/sci-fi side |
| Ben | $+6.59$ | action/sci-fi side |
| Cleo | $-7.81$ | romance/drama side |
| Dev | $-5.93$ | romance/drama side |
| Eli | $-0.18$ | neither: watches both evenly |
| Fay | $-0.37$ | neither: watches both evenly |

A coordinate near zero is not "low activity" (Fay is the heaviest viewer). It means __the contrast does not apply__: that row's positive and negative contributions cancel. Volume and preference are separate axes, and the SVD keeps them separate.

## What the second term adds

The rank-2 approximation $A^{(2)} = \sigma_1\mathbf{u}_1\mathbf{v}_1^T + \sigma_2\mathbf{u}_2\mathbf{v}_2^T$ adds the contrast to the volume. For Ana, $\sigma_2 u_{\text{Ana},2} > 0$ and $\mathbf{v}_2$ is positive on action, so the second term *raises* her action and sci-fi hours and *lowers* her romance and drama hours from the flat rank-1 row. For Cleo it does the opposite.

In [ ]:
#| code-fold: false
A2 = u[:, :2] @ np.diag(s[:2]) @ vt[:2]
pd.DataFrame(A2, index=viewers, columns=genres).round(1)

This is close to $A$ everywhere, as $98.5\%$ predicted. The error, $\Vert A - A^{(2)}\Vert_F^2 = \sigma_3^2 + \sigma_4^2 = 8.2$ hours², is the variation that neither volume nor genre preference explains, such as Ana watching more action than sci-fi.

# The Latent-Space Picture

The two coordinates of each viewer, row $i$ of $U'\Sigma'$, place them in a plane. This is the small, checkable version of the Netflix latent-space figure in Lecture 12.

In [ ]:
#| fig-align: center
#| fig-cap: "Each viewer at (σ₁u_i1, σ₂u_i2). The horizontal axis is weighted volume; the vertical axis is the action/sci-fi versus romance/drama contrast."
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.scatter(US_k.iloc[:, 0], US_k.iloc[:, 1], s=60)
for name, (x, y) in US_k.iterrows():
    offset = (-30, -14) if name == 'Ben' else (6, 4)   # Ana and Ben nearly coincide
    ax.annotate(name, (x, y), xytext=offset, textcoords='offset points', fontsize=12)
ax.axhline(0, color='gray', lw=0.8)
ax.set_xlim(0, 10.5)
ax.set_ylim(-9.5, 9.5)
ax.set_xlabel(r'$\sigma_1 u_{i1}$: overall viewing (weighted by $\mathbf{v}_1$)')
ax.set_ylabel(r'$\sigma_2 u_{i2}$: action/sci-fi $-$ romance/drama')
plt.tight_layout()
plt.show()

Read it as two independent questions. *How far right?* How much the viewer watches. *How far up or down?* Which kind of thing they watch. Eli and Fay sit on the zero line at very different distances from the left: same taste, different volume. Ana and Cleo sit at nearly the same horizontal position on opposite sides: same volume, opposite taste.

# When the Groups Are Unequal

In the table above, the two taste groups watch about the same amount, so $\mathbf{v}_1$ came out close to uniform. Change that: suppose Cleo and Dev watch half as much.

| | action | sci-fi | romance | drama | total |
|:--|:--:|:--:|:--:|:--:|:--:|
| Ana | 9 | 6 | 1 | 0 | 16 |
| Ben | 7 | 8 | 0 | 1 | 16 |
| Cleo | 1 | 0 | 5 | 4 | 10 |
| Dev | 0 | 1 | 4 | 4 | 9 |
| Eli | 2 | 2 | 2 | 2 | 8 |
| Fay | 5 | 4 | 5 | 4 | 18 |

In [ ]:
#| code-fold: false
B = A.copy()
B[2] = [1, 0, 5, 4]
B[3] = [0, 1, 4, 4]

ub, sb, vbt = np.linalg.svd(B, full_matrices=False)
for i in range(len(sb)):
    if vbt[i, 0] < 0:
        vbt[i] *= -1
        ub[:, i] *= -1

print("fraction of ||B||_F^2, cum:", np.cumsum(sb**2) / np.sum(sb**2))
print(pd.DataFrame(vbt[:2].T, index=genres, columns=['v1', 'v2']).round(2))
print(pd.DataFrame({'sigma1 u1': sb[0] * ub[:, 0], 'sigma2 u2': sb[1] * ub[:, 1],
                    'total_hours': B.sum(axis=1)}, index=viewers).round(2))

$\mathbf{v}_1 = (0.70, 0.61, 0.28, 0.24)$ is still all positive, still a "typical row", but it now leans hard toward action and sci-fi, because the heaviest viewers are action fans and $\mathbf{v}_1$ is pulled toward the rows with the most weight. The first coordinate is still the weighted total $\mathbf{b}_i\cdot\mathbf{v}_1$, but the weights are now far from equal, and the ordering by first coordinate departs from the ordering by hours:

| | total hours | $\sigma_1 u_{i1}$ |
|:--|:--:|:--:|
| Fay | 18 | 8.31 |
| Ana | 16 | 10.25 |
| Cleo | 10 | 3.07 |
| Eli | 8 | 3.66 |

Fay watches the most but Ana has the largest first coordinate; Cleo watches more than Eli but scores lower. The first coordinate measures "how much of the *typical* row", and the typical row is defined by the data, not chosen to be an even average.

The second pair is still a contrast, $\mathbf{v}_2 = (0.27, 0.25, -0.70, -0.61)$, and still splits Ana and Ben from Cleo and Dev. But now Eli ($-1.58$) and Fay ($-3.58$) are no longer on the zero line. Because $\mathbf{v}_1$ over-weights action, the first term alone predicts too much action and too little romance for an even viewer, and the second term corrects for it by pulling them toward the romance side. The rank-2 approximation is still excellent ($98.4\%$); what changed is how the two terms split the work. So: the signs of $\mathbf{u}_2$ separate the clear groups reliably, but how close to zero a "mixed" row lands depends on how balanced the table is.

# Connection to PCA

The first latent factor of a nonnegative table almost always measures "volume". Often that is not interesting: you already know some viewers watch more than others, and you want the structure *beyond* that. Lecture 13's PCA removes it by __centering__: subtract each column's mean before taking the SVD. The centered rows are no longer in the positive corner, so the argument that made $\mathbf{v}_1$ positive no longer applies.

In [ ]:
#| code-fold: false
X = A - A.mean(axis=0)
ux, sx, vxt = np.linalg.svd(X, full_matrices=False)
if vxt[0, 0] < 0:
    vxt[0] *= -1
    ux[:, 0] *= -1
print("fraction of ||X||_F^2:", sx**2 / np.sum(sx**2))
print("first right singular vector of centered X:", vxt[0])

After centering, the top direction is the action/sci-fi versus romance/drama contrast, carrying $88\%$ of what is left, and the volume difference between Eli and Fay drops to a minor second component. This is why principal components usually have mixed signs while the first singular vector of raw nonnegative data does not.

# A Checklist for Reading Singular Vectors

Given a labeled table $A$ and its first few singular vectors:

1. __Label them.__ Write the column names next to every $\mathbf{v}_k$ and the row names next to every $\mathbf{u}_k$.
2. __Fix the signs__ by a stated convention, and remember a pair $(\mathbf{u}_k, \mathbf{v}_k)$ flips together.
3. __Read $\mathbf{v}_1$.__ For nonnegative data all its entries share a sign: it is the typical row. Note which columns it weights most.
4. __Read $\mathbf{u}_1$__ (or $\sigma_1\mathbf{u}_1$) as how much of the typical row each row has: $\sigma_1 u_{i1} = \mathbf{a}_i\cdot\mathbf{v}_1$, a weighted total. Rank rows by this, not by raw totals.
5. __Read $\mathbf{v}_2$ by sign.__ Group the positive columns and the negative columns, and name the contrast. Entries near zero sit out.
6. __Read $\mathbf{u}_2$ by sign.__ Positive rows lean toward the positive columns of $\mathbf{v}_2$, negative rows toward the negative ones, rows near zero lean neither way.
7. __Check how much is explained.__ $\sum_{i\le k}\sigma_i^2/\Vert A\Vert_F^2$ near 1 means the first $k$ terms describe the table; the leftover is $\sum_{i>k}\sigma_i^2$.